# A6 — cohort and patient payloads

Split so the control board can switch *k* without refetching the patient.
Encoder identity is required. `assert_safe` runs on every generated string.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
V3 = INTERIM / "v3"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures" / "v3"
for d in (RAW, INTERIM, V3, REF, ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

SMOKE_TEST = False

def cohort_ids():
    import pandas as pd
    assign = V3 / "cluster_assignments.parquet"
    if assign.is_file():
        return pd.read_parquet(assign)["patient_id"].astype(str).str[:12].unique().tolist()
    expr = INTERIM / "intrinsic_expression.parquet"
    if expr.is_file():
        return pd.read_parquet(expr).index.astype(str).str[:12].unique().tolist()
    return None

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    if "sample_ids" not in kwargs:
        ids = cohort_ids()
        if ids is not None:
            kwargs["sample_ids"] = ids
            kwargs.setdefault("n", len(ids))
            kwargs.setdefault("cohort", True)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
from v3_payload import SCHEMA_VERSION, assert_payload_safe, copy_payloads_to_app, validate_cohort, validate_patient, v3_interim, glossary_allows_nll
from v3_real import persist_real
import json

out = persist_real(V2_ROOT, REPO_ROOT, n_boot=8 if SMOKE_TEST else 50, n_init=3 if SMOKE_TEST else 10)
cohort = json.loads(Path(out["cohort"]).read_text())
patients = {p.stem.replace("payload_", ""): json.loads(p.read_text()) for p in Path(out["cohort"]).parent.glob("payload_*.json")}
encoder = cohort["encoder"]
print("encoder", encoder, "n", cohort.get("n_samples"), "synthetic", cohort.get("synthetic_samples"), out["provenance"])


In [ ]:
from pathlib import Path
n_ok = 0
ids = list((cohort.get("configurations") or {}).get(next(iter(cohort.get("configurations") or {}), ""), {}).get("assignments") or {}).keys() or list(patients)
for path in [V3 / "cohort_payload.json", *V3.glob("payload_*.json")]:
    obj = json.loads(path.read_text())
    assert_safe(json.dumps(obj), context=str(path.name))
    n_ok += 1
gate("NB_A6", "payload_safety", float(n_ok), 1.0, cohort=True, sample_ids=ids, n=len(ids),
     note=f"assert_safe passed on {n_ok} payload files source={cohort.get('cohort_source')}")
